In [1]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from translation_dataset import create_dataloader, TranslationDataset
from prefix_t5_model import PrefixT5
from trainer import Trainer, TrainingHyperParams
from configs import CHECKPOINT, DATASET, BATCH_SIZE, MAX_LENGTH

/Users/zeinab/mambaforge/envs/llm_prefix_tuning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
t5_tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
t5_model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


## Data Preprocessing

In [3]:
# Load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Split the dataset into train, valid, test (train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])

In [4]:
train_translation_dataset = TranslationDataset(
    data=train_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

valid_translation_dataset = TranslationDataset(
    data=valid_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

test_translation_dataset = TranslationDataset(
    data=test_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)


In [5]:
train_loader = create_dataloader(
    dataset=train_translation_dataset, 
    batch_size=BATCH_SIZE
)

valid_loader = create_dataloader(
    dataset=valid_translation_dataset, 
    batch_size=BATCH_SIZE
)

test_loader = create_dataloader(
    dataset=test_translation_dataset, 
    batch_size=BATCH_SIZE, 
)

In [6]:
print(f"Number of batches in train: {len(train_loader)}")
print(f"Number of batches in valid: {len(valid_loader)}")
print(f"Number of batches in test: {len(test_loader)}")

Number of batches in train: 6434
Number of batches in valid: 715
Number of batches in test: 795


In [7]:
# Get a sample batch
sample_batch = next(iter(train_loader))
    
print("\nSample batch shapes:")
for key, value in sample_batch.items():
    print(f"{key}: {value.shape}")

# Decode a sample from the batch
print("\n")
print("Input:", t5_tokenizer.decode(sample_batch['input_ids'][0].tolist(), skip_special_tokens=True))
print("Target:", t5_tokenizer.decode(sample_batch['labels'][0].tolist(), skip_special_tokens=True))


Sample batch shapes:
input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16, 128])
decoder_input_ids: torch.Size([16, 128])


Input: translate English to French: "No," replied the two Musketeers, after a moment’s silence, "no, sir, we do not."
Target: «Savez-vous ce que m'a dit le roi, s'écria-t-il, et cela pas plus tard qu'hier au soir? le savez-vous, messieurs?


## Prefix T5 Model

In [8]:
prefixed_t5_model = PrefixT5(
    t5_model=t5_model, prefix_length=10, prefix_projection_size=512
)

## Training

In [10]:
sample_data = dataset.select(range(50))
sample_valid = dataset.select(range(50, 70))
sample_trans_data = TranslationDataset(
    data=sample_data,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_loader = create_dataloader(sample_trans_data)

sample_valid_trans_data = TranslationDataset(
    data=sample_valid,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_valid_loader = create_dataloader(sample_valid_trans_data)

In [11]:
training_params = TrainingHyperParams()
trainer = Trainer(
    model=prefixed_t5_model,
    tokenizer=t5_tokenizer,
    train_loader=sample_loader,
    valid_loader=sample_valid_loader,
    test_loader=test_loader,
    args=training_params
)

/Users/zeinab/mambaforge/envs/llm_prefix_tuning/lib/python3.11/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [12]:
trainer.train()

Validation: 100%|██████████| 2/2 [00:00<00:00,  2.38batch/s]


Epoch 1/10
	Train Loss: 2.7089
	Validation Loss: 1.1237

Model saved at models/model.pth


Validation: 100%|██████████| 2/2 [00:00<00:00,  2.50batch/s]


Epoch 2/10
	Train Loss: 2.6405
	Validation Loss: 1.2146



Validation: 100%|██████████| 2/2 [00:00<00:00,  2.48batch/s]


KeyboardInterrupt: 